## imports + chemin

In [2]:
import os
import gc
import numpy as np
import pandas as pd
from pandas.api.types import is_string_dtype

# Chemin vers dossier dataset
DATA_DIR = "/Users/macbook/OpenClassrooms/Projet7/Projet+Mise+en+prod+-+home-credit-default-risk"

## fonctions

In [4]:
def load_csv(name, nrows=None):
    return pd.read_csv(os.path.join(DATA_DIR, name), nrows=nrows)


def one_hot_encoder(df, nan_as_category=True):
    # Convert pandas string dtype -> object
    for c in df.columns:
        if is_string_dtype(df[c]):
            df[c] = df[c].astype("object")

    cat_cols = [c for c in df.columns if df[c].dtype == "object"]
    df = pd.get_dummies(df, columns=cat_cols, dummy_na=nan_as_category)
    return df


def application_base(nrows=None):
    train = load_csv("application_train.csv", nrows=nrows)
    test = load_csv("application_test.csv", nrows=nrows)

    df = pd.concat([train, test], axis=0, ignore_index=True)
    df = df[df["CODE_GENDER"] != "XNA"]

    # Remplacement anomalie
    df["DAYS_EMPLOYED"].replace(365243, np.nan, inplace=True)

    # Ratios métier
    df["DAYS_EMPLOYED_PERC"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
    df["INCOME_CREDIT_PERC"] = df["AMT_INCOME_TOTAL"] / df["AMT_CREDIT"]
    df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]
    df["ANNUITY_INCOME_PERC"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["PAYMENT_RATE"] = df["AMT_ANNUITY"] / df["AMT_CREDIT"]

    # Encodage
    df = one_hot_encoder(df, nan_as_category=True)
    gc.collect()
    return df


def bureau_and_balance(nrows=None):
    bureau = load_csv("bureau.csv", nrows=nrows)
    bb = load_csv("bureau_balance.csv", nrows=nrows)

    bb = one_hot_encoder(bb, nan_as_category=True)
    bureau = one_hot_encoder(bureau, nan_as_category=True)

    # Agrégation bb -> par SK_ID_BUREAU
    num_for_mean = [
        c for c in bb.columns
        if c not in ["SK_ID_BUREAU", "MONTHS_BALANCE"]
        and pd.api.types.is_numeric_dtype(bb[c])
    ]

    bb_aggregations = {"MONTHS_BALANCE": ["min", "max", "size"]}
    for c in num_for_mean:
        bb_aggregations[c] = ["mean"]

    bb_agg = bb.groupby("SK_ID_BUREAU").agg(bb_aggregations)
    bb_agg.columns = [f"{a}_{b.upper()}" for a, b in bb_agg.columns]

    # Agrégation bureau -> par SK_ID_CURR
    num_cols = [
        c for c in bureau.columns
        if c not in ["SK_ID_CURR"]
        and bureau[c].dtype != "uint8"
        and bureau[c].dtype != "int8"
    ]
    cat_cols = [
        c for c in bureau.columns
        if c not in ["SK_ID_CURR"]
        and (str(bureau[c].dtype).startswith("uint") or str(bureau[c].dtype).startswith("int8"))
    ]

    num_aggs = {c: ["min", "max", "mean", "sum"] for c in num_cols}
    cat_aggs = {c: ["mean"] for c in cat_cols}

    buro_agg = bureau.groupby("SK_ID_CURR").agg({**num_aggs, **cat_aggs})
    buro_agg.columns = [f"BURO_{a}_{b.upper()}" for a, b in buro_agg.columns]

    # Actifs
    if "CREDIT_ACTIVE_Active" in bureau.columns:
        active = bureau[bureau["CREDIT_ACTIVE_Active"] == 1]
        active_agg = active.groupby("SK_ID_CURR").agg(num_aggs)
        active_agg.columns = [f"ACTIVE_{a}_{b.upper()}" for a, b in active_agg.columns]
        buro_agg = buro_agg.join(active_agg, how="left")

    # Clos
    if "CREDIT_ACTIVE_Closed" in bureau.columns:
        closed = bureau[bureau["CREDIT_ACTIVE_Closed"] == 1]
        closed_agg = closed.groupby("SK_ID_CURR").agg(num_aggs)
        closed_agg.columns = [f"CLOSED_{a}_{b.upper()}" for a, b in closed_agg.columns]
        buro_agg = buro_agg.join(closed_agg, how="left")

    del bureau, bb, bb_agg
    gc.collect()
    return buro_agg


def previous_applications(nrows=None):
    prev = load_csv("previous_application.csv", nrows=nrows)
    prev = one_hot_encoder(prev, nan_as_category=True)

    for c in [
        "DAYS_FIRST_DRAWING",
        "DAYS_FIRST_DUE",
        "DAYS_LAST_DUE_1ST_VERSION",
        "DAYS_LAST_DUE",
        "DAYS_TERMINATION"
    ]:
        if c in prev.columns:
            prev[c].replace(365243, np.nan, inplace=True)

    prev["APP_CREDIT_PERC"] = prev["AMT_APPLICATION"] / prev["AMT_CREDIT"]

    num_cols = [
        "AMT_ANNUITY",
        "AMT_APPLICATION",
        "AMT_CREDIT",
        "APP_CREDIT_PERC",
        "AMT_DOWN_PAYMENT",
        "AMT_GOODS_PRICE",
        "DAYS_DECISION",
        "CNT_PAYMENT"
    ]
    num_cols = [c for c in num_cols if c in prev.columns]

    num_aggs = {c: ["min", "max", "mean"] for c in num_cols}
    if "APP_CREDIT_PERC" in num_aggs:
        num_aggs["APP_CREDIT_PERC"] = ["min", "max", "mean", "var"]

    cat_cols = [
        c for c in prev.columns
        if c not in ["SK_ID_CURR", "SK_ID_PREV"] + num_cols
        and prev[c].dtype in [np.uint8, np.int8, np.int16, np.int32, np.int64]
    ]
    cat_aggs = {c: ["mean"] for c in cat_cols}

    prev_agg = prev.groupby("SK_ID_CURR").agg({**num_aggs, **cat_aggs})
    prev_agg.columns = [f"PREV_{a}_{b.upper()}" for a, b in prev_agg.columns]

    if "NAME_CONTRACT_STATUS_Approved" in prev.columns:
        approved = prev[prev["NAME_CONTRACT_STATUS_Approved"] == 1]
        approved_agg = approved.groupby("SK_ID_CURR").agg(num_aggs)
        approved_agg.columns = [f"APPROVED_{a}_{b.upper()}" for a, b in approved_agg.columns]
        prev_agg = prev_agg.join(approved_agg, how="left")

    if "NAME_CONTRACT_STATUS_Refused" in prev.columns:
        refused = prev[prev["NAME_CONTRACT_STATUS_Refused"] == 1]
        refused_agg = refused.groupby("SK_ID_CURR").agg(num_aggs)
        refused_agg.columns = [f"REFUSED_{a}_{b.upper()}" for a, b in refused_agg.columns]
        prev_agg = prev_agg.join(refused_agg, how="left")

    del prev
    gc.collect()
    return prev_agg


def pos_cash(nrows=None):
    pos = load_csv("POS_CASH_balance.csv", nrows=nrows)
    pos = one_hot_encoder(pos, nan_as_category=True)

    aggs = {
        "MONTHS_BALANCE": ["max", "mean", "size"],
        "SK_DPD": ["max", "mean"],
        "SK_DPD_DEF": ["max", "mean"]
    }

    pos_agg = pos.groupby("SK_ID_CURR").agg(aggs)
    pos_agg.columns = [f"POS_{a}_{b.upper()}" for a, b in pos_agg.columns]
    pos_agg["POS_COUNT"] = pos.groupby("SK_ID_CURR").size()

    del pos
    gc.collect()
    return pos_agg


def installments_payments(nrows=None):
    ins = load_csv("installments_payments.csv", nrows=nrows)
    ins = one_hot_encoder(ins, nan_as_category=True)

    ins["PAYMENT_PERC"] = ins["AMT_PAYMENT"] / ins["AMT_INSTALMENT"]
    ins["PAYMENT_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]
    ins["DPD"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0)
    ins["DBD"] = (ins["DAYS_INSTALMENT"] - ins["DAYS_ENTRY_PAYMENT"]).clip(lower=0)

    aggs = {
        "DPD": ["max", "mean", "sum"],
        "DBD": ["max", "mean", "sum"],
        "PAYMENT_PERC": ["max", "mean", "var"],
        "PAYMENT_DIFF": ["max", "mean", "sum", "var"],
        "AMT_INSTALMENT": ["max", "mean", "sum"],
        "AMT_PAYMENT": ["min", "max", "mean", "sum"],
    }

    ins_agg = ins.groupby("SK_ID_CURR").agg(aggs)
    ins_agg.columns = [f"INSTAL_{a}_{b.upper()}" for a, b in ins_agg.columns]
    ins_agg["INSTAL_COUNT"] = ins.groupby("SK_ID_CURR").size()

    del ins
    gc.collect()
    return ins_agg


def credit_card_balance(nrows=None):
    cc = load_csv("credit_card_balance.csv", nrows=nrows)
    cc = one_hot_encoder(cc, nan_as_category=True)

    if "SK_ID_PREV" in cc.columns:
        cc.drop(columns=["SK_ID_PREV"], inplace=True)

    cc_agg = cc.groupby("SK_ID_CURR").agg(["min", "max", "mean", "sum"])
    cc_agg.columns = [f"CC_{a}_{b.upper()}" for a, b in cc_agg.columns]
    cc_agg["CC_COUNT"] = cc.groupby("SK_ID_CURR").size()

    del cc
    gc.collect()
    return cc_agg


def build_features(nrows=None):
    df = application_base(nrows=nrows)

    buro = bureau_and_balance(nrows=nrows)
    df = df.merge(buro, on="SK_ID_CURR", how="left")
    del buro
    gc.collect()

    prev = previous_applications(nrows=nrows)
    df = df.merge(prev, on="SK_ID_CURR", how="left")
    del prev
    gc.collect()

    pos = pos_cash(nrows=nrows)
    df = df.merge(pos, on="SK_ID_CURR", how="left")
    del pos
    gc.collect()

    ins = installments_payments(nrows=nrows)
    df = df.merge(ins, on="SK_ID_CURR", how="left")
    del ins
    gc.collect()

    cc = credit_card_balance(nrows=nrows)
    df = df.merge(cc, on="SK_ID_CURR", how="left")
    del cc
    gc.collect()

    return df

## test rapide

In [6]:
df = build_features(nrows=None)
print(df.shape)
df.head()

/var/folders/6q/vqmkpd597p36s9t_j9zm_k0c0000gn/T/ipykernel_22570/3264766248.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["DAYS_EMPLOYED"].replace(365243, np.nan, inplace=True)
/var/folders/6q/vqmkpd597p36s9t_j9zm_k0c0000gn/T/ipykernel_22570/3264766248.py:110: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting

(356251, 954)


,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,CC_NAME_CONTRACT_STATUS_Sent proposal_SUM,CC_NAME_CONTRACT_STATUS_Signed_MIN,CC_NAME_CONTRACT_STATUS_Signed_MAX,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_Signed_SUM,CC_NAME_CONTRACT_STATUS_nan_MIN,CC_NAME_CONTRACT_STATUS_nan_MAX,CC_NAME_CONTRACT_STATUS_nan_MEAN,CC_NAME_CONTRACT_STATUS_nan_SUM,CC_COUNT
0,100002,1.0,0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461,-637.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0.0,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765,-1188.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0.0,0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046,-225.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0.0,0,135000.0,312682.5,29686.5,297000.0,0.008019,-19005,-3039.0,...,0.0,False,False,0.0,0.0,False,False,0.0,0.0,6.0
4,100007,0.0,0,121500.0,513000.0,21865.5,513000.0,0.028663,-19932,-3038.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## séparation train / test

In [8]:
train_df = df[df["TARGET"].notnull()].copy()
test_df = df[df["TARGET"].isnull()].copy()

print("train_df :", train_df.shape)
print("test_df  :", test_df.shape)
print("SK_ID_CURR dans test_df :", "SK_ID_CURR" in test_df.columns)
print("TARGET dans test_df :", "TARGET" in test_df.columns)

train_df : (307507, 954)
test_df  : (48744, 954)
SK_ID_CURR dans test_df : True
TARGET dans test_df : True


## créer le dataset API

In [10]:
id_col = "SK_ID_CURR"

df_api = test_df.drop(columns=["TARGET"]).copy()
df_api_100 = df_api.sample(n=100, random_state=42).copy()
X_api_100 = df_api_100.drop(columns=[id_col]).copy()

print("df_api_100 :", df_api_100.shape)
print("X_api_100  :", X_api_100.shape)
print("ID présent dans X_api_100 ?", id_col in X_api_100.columns)

df_api_100 : (100, 953)
X_api_100  : (100, 952)
ID présent dans X_api_100 ? False


## sauvegarde

In [12]:
os.makedirs("../api/data", exist_ok=True)

df_api_100.to_csv("../api/data/test_clients.csv", index=False)
X_api_100.to_csv("../api/data/test_clients_features_only.csv", index=False)

print("Fichiers sauvegardés avec succès")

Fichiers sauvegardés avec succès


## vérification

In [27]:
df_check = pd.read_csv("/Users/macbook/OpenClassrooms/api/data/test_clients.csv")
X_check = pd.read_csv("/Users/macbook/OpenClassrooms/api/data/test_clients_features_only.csv")

print("test_clients.csv :", df_check.shape)
print("test_clients_features_only.csv :", X_check.shape)
print("Nb clients uniques :", df_check["SK_ID_CURR"].nunique())

test_clients.csv : (100, 953)
test_clients_features_only.csv : (100, 952)
Nb clients uniques : 100


## charger le modèle et les features

In [30]:
import joblib
import pandas as pd

model = joblib.load("/Users/macbook/OpenClassrooms/Projet7/model.joblib")
X_api_100 = pd.read_csv("/Users/macbook/OpenClassrooms/api/data/test_clients_features_only.csv")

print("Shape X_api_100 :", X_api_100.shape)
print("Type du modèle :", type(model))

Shape X_api_100 : (100, 952)
Type du modèle : <class 'xgboost.sklearn.XGBClassifier'>


In [32]:
obj_cols = X_api_100.select_dtypes(include=["object"]).columns.tolist()

print("Nombre de colonnes object :", len(obj_cols))
print(obj_cols[:20])

Nombre de colonnes object : 172
['BURO_CREDIT_ACTIVE_Active_MIN', 'BURO_CREDIT_ACTIVE_Active_MAX', 'BURO_CREDIT_ACTIVE_Bad debt_MIN', 'BURO_CREDIT_ACTIVE_Bad debt_MAX', 'BURO_CREDIT_ACTIVE_Closed_MIN', 'BURO_CREDIT_ACTIVE_Closed_MAX', 'BURO_CREDIT_ACTIVE_Sold_MIN', 'BURO_CREDIT_ACTIVE_Sold_MAX', 'BURO_CREDIT_ACTIVE_nan_MIN', 'BURO_CREDIT_ACTIVE_nan_MAX', 'BURO_CREDIT_CURRENCY_currency 1_MIN', 'BURO_CREDIT_CURRENCY_currency 1_MAX', 'BURO_CREDIT_CURRENCY_currency 2_MIN', 'BURO_CREDIT_CURRENCY_currency 2_MAX', 'BURO_CREDIT_CURRENCY_currency 3_MIN', 'BURO_CREDIT_CURRENCY_currency 3_MAX', 'BURO_CREDIT_CURRENCY_currency 4_MIN', 'BURO_CREDIT_CURRENCY_currency 4_MAX', 'BURO_CREDIT_CURRENCY_nan_MIN', 'BURO_CREDIT_CURRENCY_nan_MAX']


In [34]:
for col in obj_cols[:10]:
    print(f"\n--- {col} ---")
    print(X_api_100[col].dropna().unique()[:10])


--- BURO_CREDIT_ACTIVE_Active_MIN ---
[False True]

--- BURO_CREDIT_ACTIVE_Active_MAX ---
[True False]

--- BURO_CREDIT_ACTIVE_Bad debt_MIN ---
[False]

--- BURO_CREDIT_ACTIVE_Bad debt_MAX ---
[False]

--- BURO_CREDIT_ACTIVE_Closed_MIN ---
[False True]

--- BURO_CREDIT_ACTIVE_Closed_MAX ---
[True False]

--- BURO_CREDIT_ACTIVE_Sold_MIN ---
[False]

--- BURO_CREDIT_ACTIVE_Sold_MAX ---
[False True]

--- BURO_CREDIT_ACTIVE_nan_MIN ---
[False]

--- BURO_CREDIT_ACTIVE_nan_MAX ---
[False]


In [36]:
X_api_100_fixed = X_api_100.copy()

obj_cols = X_api_100_fixed.select_dtypes(include=["object"]).columns.tolist()
print("Colonnes object avant conversion :", len(obj_cols))

for col in obj_cols:
    # tentative conversion directe
    X_api_100_fixed[col] = pd.to_numeric(X_api_100_fixed[col], errors="coerce")

print("\nTypes après conversion :")
print(X_api_100_fixed.dtypes.value_counts())

remaining_obj_cols = X_api_100_fixed.select_dtypes(include=["object"]).columns.tolist()
print("\nColonnes object restantes :", len(remaining_obj_cols))
print(remaining_obj_cols[:20])

Colonnes object avant conversion : 172

Types après conversion :
float64    759
bool       155
int64       38
Name: count, dtype: int64

Colonnes object restantes : 0
[]


In [38]:
print("Nombre total de NaN :", X_api_100_fixed.isna().sum().sum())
print("Shape :", X_api_100_fixed.shape)

Nombre total de NaN : 23962
Shape : (100, 952)


In [40]:
import joblib

model = joblib.load("/Users/macbook/OpenClassrooms/Projet7/model.joblib")

X_one = X_api_100_fixed.iloc[[0]]

pred = model.predict(X_one)
proba = model.predict_proba(X_one)

print("Prediction :", pred)
print("Probabilité :", proba)

ValueError: Feature shape mismatch, expected: 953, got 952

In [42]:
df_with_id = pd.read_csv("../api/data/test_clients.csv")
X_without_id = pd.read_csv("../api/data/test_clients_features_only.csv")

print("Avec ID :", df_with_id.shape)
print("Sans ID :", X_without_id.shape)

Avec ID : (100, 953)
Sans ID : (100, 952)


In [44]:
df_with_id_fixed = df_with_id.copy()

for col in df_with_id_fixed.select_dtypes(include=["object"]).columns:
    df_with_id_fixed[col] = pd.to_numeric(df_with_id_fixed[col], errors="coerce")

print(df_with_id_fixed.shape)
print(df_with_id_fixed.dtypes.value_counts())

(100, 953)
float64    759
bool       155
int64       39
Name: count, dtype: int64


In [46]:
X_one_with_id = df_with_id_fixed.iloc[[0]].copy()

pred = model.predict(X_one_with_id)
proba = model.predict_proba(X_one_with_id)

print("Prediction :", pred)
print("Probabilité :", proba)

Prediction : [0]
Probabilité : [[0.983722   0.01627803]]
